# FACT Reproduction: NF0201train

This notebook runs the self-contained NeuroFinder reproduction, records its execution outputs, computes background-removed traces, and provides an interactive mask/trace viewer.

Data required: TIFF: data\NeuroFinder-Registered\0201_train\NF0201train.tiff

In [ ]:
# All resources are resolved below this notebook's FACT-Code root
import gc
import sys
import time
from pathlib import Path


def _find_release_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        required = (
            candidate / "IO",
            candidate / "ModelInference",
            candidate / "PostSlice",
            candidate / "model",
            candidate / "ModelParams" / "FACT_Modelparams.pt",
            candidate / "data" / "NeuroFinder-Registered",
        )
        if all(path.exists() for path in required):
            return candidate
    raise RuntimeError(
        "Run this notebook with the FACT-Code directory as the working "
        "directory (or a child directory of it)."
    )


PROJECT_ROOT = _find_release_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import SimpleITK before the PyTorch/MONAI stack on Windows.
import SimpleITK as sitk  # noqa: F401
import numpy as np
import torch

from IO.Read_tif import load_tiff
from IO.ReadmatMask import read_mat_mask
from ModelInference.SWInf import sliding_window_inference
from PostSlice.Evaluation.Jaccard_sparse import evaluation_jaccard
from PostSlice.PostProcess_Pipeline import (
    Post_FACT_guided_backgroundremoving,
    extract_roi_mean_traces,
    Post_standard_spatial_consolidation,
)
from model.TS_Net_change import FACT_Net

# The imports above must resolve inside this Release tree.
import IO
import ModelInference
import PostSlice
import model

for _module in (IO, ModelInference, PostSlice, model):
    _module_locations = list(getattr(_module, "__path__", ()))
    if not _module_locations and getattr(_module, "__file__", None):
        _module_locations = [str(Path(_module.__file__).resolve().parent)]
    if not _module_locations:
        raise RuntimeError(f"Could not locate imported package: {_module.__name__}")
    if not all(
        Path(_location).resolve().is_relative_to(PROJECT_ROOT.resolve())
        for _location in _module_locations
    ):
        raise RuntimeError(
            f"External project import detected: {_module.__name__} -> {_module_locations}"
        )

print(f"Release root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

ROI_SIZE = (128, 64, 64)
MAX_SW_BATCH_SIZE = 32


def _is_cuda_oom(error):
    return isinstance(error, torch.cuda.OutOfMemoryError) or (
        isinstance(error, RuntimeError) and "out of memory" in str(error).lower()
    )


def _release_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_auto_sw_batch_size(
    roi_size=ROI_SIZE,
    max_batch_size=MAX_SW_BATCH_SIZE,
    reserved_ratio=0.05,
    mem_per_patch_mb=705,
    device=None,
):
    """Choose a patch batch from current free VRAM; CPU always uses one patch."""
    del roi_size  # Kept for parity with the original helper's public signature.
    target_device = (
        torch.device(device)
        if device is not None
        else torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    )
    if target_device.type != "cuda" or not torch.cuda.is_available():
        return 1

    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info(target_device)
    safety_ratio = min(max(float(reserved_ratio), 0.0), 0.95)
    usable_bytes = max(0, int(float(free_bytes) * (1.0 - safety_ratio)))
    mem_per_patch = max(1, int(mem_per_patch_mb)) * 1024 * 1024
    estimated_batch = int(usable_bytes // mem_per_patch)
    batch_size = max(1, min(int(max_batch_size), estimated_batch))
    print(
        "Auto sw_batch_size: {} (free VRAM ~{:.1f}/{:.1f} GB; safety {:.0%})".format(
            batch_size,
            float(free_bytes) / 1024**3,
            float(total_bytes) / 1024**3,
            safety_ratio,
        )
    )
    return batch_size


### Load data

In [2]:
DATASET_NAME = "NF0201train"
DATASET_SUBDIR = "0201_train"
DATA_PATH = PROJECT_ROOT / "data" / "NeuroFinder-Registered" / DATASET_SUBDIR / f"{DATASET_NAME}.tiff"
GT_PATH = PROJECT_ROOT / "data" / "NeuroFinder-Registered" / DATASET_SUBDIR / f"{DATASET_NAME}__GTmasks.mat"
MODEL_PATH = PROJECT_ROOT / "ModelParams" / "FACT_Modelparams.pt"

for _path in (DATA_PATH, GT_PATH, MODEL_PATH):
    if not _path.is_file():
        raise FileNotFoundError(_path)
print(f"Dataset: {DATASET_NAME}")
print(f"TIFF: {DATA_PATH.relative_to(PROJECT_ROOT)}")


Dataset: NF0201train
TIFF: data\NeuroFinder-Registered\0201_train\NF0201train.tiff


In [3]:
time_start = time.perf_counter()
input_img = np.asarray(load_tiff(str(DATA_PATH)), dtype=np.float32)
if input_img.ndim != 3:
    raise ValueError(f"Expected a T x H x W video, got {input_img.shape!r}.")
raw_min = float(input_img.min())
raw_max = float(input_img.max())
if not np.isfinite(raw_min) or not np.isfinite(raw_max) or raw_max <= raw_min:
    raise ValueError(f"Cannot min-max normalize input with range [{raw_min}, {raw_max}].")
input_img = (input_img - raw_min) / (raw_max - raw_min)
print(f"Loaded normalized video: shape={input_img.shape}, dtype={input_img.dtype}")
print(f"Input range before normalization: [{raw_min}, {raw_max}]")
print(f"Read/normalize seconds: {time.perf_counter() - time_start:.2f}")


Loaded normalized video: shape=(8000, 504, 464), dtype=float32
Input range before normalization: [0.0, 1.0]
Read/normalize seconds: 184.93


### Load FACT network

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Inference device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def _build_fact_model(target_device):
    fact_model = FACT_Net(
        img_size=ROI_SIZE,
        in_channels=1,
        out_channels=2,
        init_dim=3,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        use_checkpoint=True,
    ).to(target_device)
    weight = torch.load(str(MODEL_PATH), map_location="cpu")
    fact_model.load_state_dict(weight["state_dict"])
    fact_model.eval()
    return fact_model


try:
    model = _build_fact_model(device)
except RuntimeError as exc:
    if device.type != "cuda" or not _is_cuda_oom(exc):
        raise
    del exc
    print("GPU cannot hold the FACT model; falling back to CPU.")
    _release_cuda_memory()
    device = torch.device("cpu")
    model = _build_fact_model(device)

print(f"Loaded model: {MODEL_PATH.relative_to(PROJECT_ROOT)}")


### Network Inference

In [ ]:
time_start = time.perf_counter()
# Tested configuration: sw_batch_size=32 on an NVIDIA RTX 3090 with 24 GiB VRAM.
# Start at 32 to preserve the original reference execution path. Automatic
# sizing is used only after a CUDA OOM to choose a safe fallback.
sw_batch_size = MAX_SW_BATCH_SIZE
print(f"Starting network inference with sw_batch_size={sw_batch_size} on {device}")


def _run_inference_once(batch_size):
    input_img_tensor = torch.from_numpy(input_img)
    test_inputs = input_img_tensor.unsqueeze(0).unsqueeze(1)
    with torch.no_grad():
        test_outputs = sliding_window_inference(
            test_inputs,
            ROI_SIZE,
            sw_batch_size=int(batch_size),
            predictor=model,
            overlap=[0.6, 0.2, 0.2],
            progress=True,
            mode="constant",
            device=torch.device("cpu"),
            sw_device=device,
        )
        diff = test_outputs[0, 1, :, :, :] - test_outputs[0, 0, :, :, :]
        infer_mask3d_local = (diff > 0).to(dtype=torch.uint8).cpu().numpy()
    del test_outputs, diff, test_inputs, input_img_tensor
    return infer_mask3d_local


while True:
    try:
        infer_mask3d = _run_inference_once(sw_batch_size)
        break
    except RuntimeError as exc:
        if device.type != "cuda" or not _is_cuda_oom(exc):
            raise
        del exc
        _release_cuda_memory()
        if sw_batch_size > 1:
            previous_batch_size = sw_batch_size
            next_batch_size = max(1, previous_batch_size // 2)
            if previous_batch_size == MAX_SW_BATCH_SIZE:
                estimated_batch_size = get_auto_sw_batch_size(
                    roi_size=ROI_SIZE,
                    max_batch_size=MAX_SW_BATCH_SIZE,
                    reserved_ratio=0.05,
                    mem_per_patch_mb=705,
                    device=device,
                )
                next_batch_size = min(next_batch_size, estimated_batch_size)
            sw_batch_size = next_batch_size
            print(
                f"CUDA OOM at sw_batch_size={previous_batch_size}; "
                f"retrying with sw_batch_size={sw_batch_size}."
            )
            continue

        print("CUDA OOM at sw_batch_size=1; falling back to CPU inference.")
        model = model.to(torch.device("cpu"))
        device = torch.device("cpu")
        _release_cuda_memory()
        sw_batch_size = 1
        infer_mask3d = _run_inference_once(sw_batch_size)
        break

if device.type == "cuda":
    torch.cuda.empty_cache()
gc.collect()
print(f"Network inference complete: infer_mask3d={infer_mask3d.shape}")
print(f"Inference sw_batch_size used: {sw_batch_size}")
print(f"Inference device used: {device}")
print(f"Inference seconds: {time.perf_counter() - time_start:.2f}")


### Calculate STD

In [6]:
T, H, W = (int(value) for value in infer_mask3d.shape)
mask_shape_2d = (H, W)
infer_mask3d = np.asarray(infer_mask3d, dtype=np.uint8)

# Same temporal-STD display image used by the Release GUI, computed locally so
# the viewer has no dependency on any external project.
std_raw = np.std(input_img, axis=0, dtype=np.float64)
std_min = float(std_raw.min())
std_max = float(std_raw.max())
if std_max <= std_min:
    std_img = np.zeros((H, W), dtype=np.float32)
else:
    std_img = ((std_raw - std_min) / (std_max - std_min) * 255.0).astype(np.float32)
del std_raw
gc.collect()
print(f"Postprocess mask shape: {mask_shape_2d}; frames: {T}")


Postprocess mask shape: (504, 464); frames: 8000


## Post-processing 

In [7]:
POST_PARAMS = {
    "max_area": 230,
    "min_area": 30,
    "thresh_com": 4.0,
    "thresh_com_refine": 7.0,
    "thresh_iou": 0.7,
    "thresh_acti": 4,
    "thresh_refine": 0.2,
    "num_workers": 8,
}

post_start = time.perf_counter()
masks_final, _timestamps_final = Post_standard_spatial_consolidation(
    infer_mask3d,
    mask_shape_2d=mask_shape_2d,
    min_area=POST_PARAMS["min_area"],
    max_area=POST_PARAMS["max_area"],
    thresh_com=POST_PARAMS["thresh_com"],
    thresh_com_refine=POST_PARAMS["thresh_com_refine"],
    thresh_refine=POST_PARAMS["thresh_refine"],
    thresh_iou=POST_PARAMS["thresh_iou"],
    thresh_consume=(1 + POST_PARAMS["thresh_iou"]) / 2,
    thresh_acti=POST_PARAMS["thresh_acti"],
    num_workers=POST_PARAMS["num_workers"],
)
masks_final = (masks_final.tocsr() > 0).astype(np.int64)
print(f"Spatial postprocess complete: masks={masks_final.shape[0]}")
print(f"Postprocess seconds: {time.perf_counter() - post_start:.2f}")


Finding mask pairs


Calculating transfered masks


Spatial postprocess complete: masks=269
Postprocess seconds: 6.61


In [ ]:
# Initialize one ROI-mean trace per mask.
masks_for_trace = (masks_final.tocsr() > 0).astype(np.int64)
traces_roi_init = extract_roi_mean_traces(masks_for_trace, input_img)
sources = np.zeros((masks_for_trace.shape[0],), dtype=bool)

background_result = Post_FACT_guided_backgroundremoving(
    masks_for_trace,
    traces_roi_init,
    sources,
    infer_mask3d=infer_mask3d,
    input_img=input_img,
    mask_shape_2d=mask_shape_2d,
    run_overlap_demix=False,
    binarize_final=True,
)
masks_final = (background_result["masks_final"].tocsr() > 0).astype(np.int64)
traces_roi = np.asarray(background_result["traces_cell"], dtype=np.float32)
traces_background = np.asarray(background_result["traces_bg"], dtype=np.float32)
traces_final = np.asarray(background_result["traces_final"], dtype=np.float32)
sources = np.asarray(background_result["sources"], dtype=bool).reshape(-1)
alpha_nnls = np.asarray(background_result["alpha_nnls"], dtype=np.float32).reshape(-1)
del masks_for_trace, traces_roi_init, background_result
gc.collect()
print(f"FACT-guided background removal complete: masks={masks_final.shape[0]}")


FACT-guided background removal complete: masks=269


## Evaluation

In [ ]:
masks_GT = read_mat_mask(str(GT_PATH), mask_shape_2d=mask_shape_2d)
match_gt_list, precision, recall, F1 = evaluation_jaccard(
    masks_final,
    masks_GT,
    0.5,
)
print(f"F1: {F1}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")

## Interactive exhibition of masks and traces

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


def _single_mask_flat(mask_array, row_index):
    row = mask_array.getrow(int(row_index))
    return np.asarray(row.toarray()).ravel()


def _warm_color(row_index):
    # Deterministic warm hue, so re-running a cell keeps the same appearance.
    hue = (0.035 + 0.11 * ((int(row_index) * 37) % 101) / 100.0) % 1.0
    return (mcolors.hsv_to_rgb((hue, 0.90, 1.0)) * 255.0).astype(np.float32)


def _overlay_for_row(row_index):
    mask = _single_mask_flat(masks_final, row_index).reshape(mask_shape_2d) > 0
    gray = np.clip(std_img, 0.0, 255.0).astype(np.float32)
    overlay = np.repeat(gray[..., None], 3, axis=2)
    color = _warm_color(row_index)
    overlay[mask] = 0.5 * overlay[mask] + 0.5 * color
    return np.clip(overlay, 0, 255).astype(np.uint8), mask


mask_areas = np.asarray(masks_final.getnnz(axis=1)).ravel()
sorted_neuron_indices = np.argsort(mask_areas)[::-1]
neuron_count = int(masks_final.shape[0])


def show_single_neuron(sorted_idx):
    sorted_idx = int(sorted_idx)
    neuron_idx = int(sorted_neuron_indices[sorted_idx])
    overlay, mask = _overlay_for_row(neuron_idx)
    trace = np.asarray(traces_final[neuron_idx]).ravel()
    fig, axes = plt.subplots(1, 2, figsize=(16, 3.8))
    axes[0].imshow(overlay)
    axes[0].set_title(
        f"Neuron #{neuron_idx} mask on temporal-STD (area={int(mask_areas[neuron_idx])})"
    )
    axes[0].axis("off")
    axes[1].plot(trace, lw=0.9, color="#e07b00")
    axes[1].set_title(f"Background-corrected trace (sorted #{sorted_idx})")
    axes[1].set_xlabel("Frame")
    axes[1].set_ylabel("Signal")
    axes[1].grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


if neuron_count == 0:
    print("No masks were produced; interactive viewer is unavailable.")
else:
    # Render one static initial view as a durable saved notebook output, then
    # expose the same view through the CA3-style interactive slider.
    show_single_neuron(0)
    neuron_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=neuron_count - 1,
        step=1,
        description="Sorted#",
        continuous_update=False,
    )
    viewer_output = widgets.interactive_output(
        show_single_neuron, {"sorted_idx": neuron_slider}
    )
    display(widgets.VBox([neuron_slider, viewer_output]))
